In [ ]:
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
import time

In [5]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

def get_html_statusinvest(ticker, max_tentativas=3):
    driver = webdriver.Chrome()
    wait = WebDriverWait(driver, 20)

    try:
        driver.get(f"https://statusinvest.com.br/acoes/{ticker}")

        # Aguarda o carregamento do bloco de informações de setor
        wait.until(
            EC.presence_of_element_located(
                (By.XPATH, "//span[contains(text(), 'Setor de Atuação')]")
            )
        )

        print("Elemento 'Setor de Atuação' carregado.")

        html = driver.page_source
        return html

    finally:
        driver.quit()


In [6]:
ticket = "rani3"
html = get_html_statusinvest(ticket)

Elemento 'Setor de Atuação' carregado.


In [13]:
from bs4 import BeautifulSoup
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

def get_setor_html_statusinvest(ticker, max_tentativas=3):
    driver = webdriver.Chrome()
    wait = WebDriverWait(driver, 20)

    try:
        driver.get(f"https://statusinvest.com.br/acoes/{ticker}")

        # Aguarda o carregamento do bloco de informações de setor
        wait.until(
            EC.presence_of_element_located(
                (By.XPATH, "//span[contains(text(), 'Setor de Atuação')]")
            )
        )

        print("Elemento 'Setor de Atuação' carregado.")

        html = driver.page_source
        return html

    finally:
        driver.quit()


ticket = "rani3"

def setor(ticket):
    html = get_setor_html_statusinvest(ticket)

    soup = BeautifulSoup(html, "html.parser")

    data = {
        "Ticket": ticket.upper(),
        "Setor de Atuação": None,
        "Subsetor de Atuação": None,
        "Segmento de Atuação": None
    }

    for info in soup.select("div.info"):
        label_tag = info.select_one("span.sub-value")
        value_tag = info.select_one("strong.value")

        if label_tag and value_tag:
            label = label_tag.get_text(strip=True)
            value = value_tag.get_text(strip=True)
            data[label] = value

    df = pd.DataFrame([data])
    df = df[["Ticket", "Setor de Atuação", "Subsetor de Atuação", "Segmento de Atuação"]]
    return df

print(setor(ticket))

Elemento 'Setor de Atuação' carregado.
  Ticket   Setor de Atuação Subsetor de Atuação Segmento de Atuação
0  RANI3  Materiais Básicos          Embalagens          Embalagens


In [ ]:
def get_html_statusinvest(ticker):

    driver = webdriver.Chrome()
    driver.get(f"https://statusinvest.com.br/acoes/{ticker}")

    print("Página carregada.")

    try:
        # aguardar até o botão aparecer e ficar clicável (máx 20 segundos)
        btn = WebDriverWait(driver, 20).until(
            EC.element_to_be_clickable((By.XPATH, "//button[@title='Histórico do ativo']"))
        )
        
        # rolar até o botão ficar visível
        driver.execute_script("arguments[0].scrollIntoView(true);", btn)
        
        # rolar mais 200px para cima
        driver.execute_script("window.scrollBy(0, -200);")
        
        # clicar
        btn.click()
        print("Botão HISTÓRICO clicado com sucesso.")
        
        # aguardar até a tabela histórica aparecer (máx 20 segundos)
        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "div.table-history"))
        )
        print("Tabela histórica carregada.")

    except Exception as e:
        print("Erro ao tentar clicar no botão HISTÓRICO:", e)

    # capturar HTML atualizado
    html = driver.page_source
    driver.quit()
    
    return html